# ICF Prediction Value Analysis
## Three research questions:
1. **Does the physiotherapist predict complications well?** (physio as a standalone classifier)
2. **Are ICF-available patients a biased subset?** (selection bias check)
3. **Do preoperative ICF scores add predictive value** beyond the staged ML model?

---
**Cohort:** DUCA 2016–2025, N=1,181 esophageal cancer patients  
**ICF subset:** N≈533 with at least one ICF note in the 30 days before surgery  
**Outcome:** `compl` — any postoperative complication within 30 days

## 0. Configuration — edit paths here

In [ ]:
import os

# ── PATHS ─────────────────────────────────────────────────────────────────────
# NOTE: source data (DUCA registry extracts, ICF notes) is not included in this
# public repo for patient privacy reasons. Update these paths to point at your
# local copies of the source files before running.
PHYSIO_PATH   = "../data/ICF_patients_without_valid_assessment_in_DUCA_or_DLCA_ege.csv"
ICF_PERI_PATH = "../data/icf_perioperative.csv"
SURG_PATH     = "../data/oesofagus_surgical.csv"
PREPROCESSED_DIR = None

# Column names
PATIENT_ID   = "MDN"
SURG_ID_COL  = "upn"
OUTCOME_COL  = "compl"
PHYSIO_RISK  = "high / low"
DIAGNOSE_COL = "diagnose"

RANDOM_SEED = 42
N_FOLDS     = 5
# ─────────────────────────────────────────────────────────────────────────────

print("Config OK")
for label, path in [("Physio file", PHYSIO_PATH), ("ICF peri", ICF_PERI_PATH), ("Surgical", SURG_PATH)]:
    exists = os.path.exists(path)
    print(f"  {label}: {'✓ found' if exists else '✗ NOT FOUND — check path'}")

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    average_precision_score, confusion_matrix,
    balanced_accuracy_score
)
from sklearn.metrics import make_scorer

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — skipping XGB")

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

print("Imports OK")

## 2. Load and merge data

In [ ]:
# ── Load physiotherapist conclusions ──────────────────────────────────────────
if PHYSIO_PATH.endswith('.csv'):
    physio = pd.read_csv(PHYSIO_PATH, sep=None, engine='python')
else:
    physio = pd.read_excel(PHYSIO_PATH)

print(f"Physio file: {physio.shape[0]} rows x {physio.shape[1]} columns")
print(f"Columns: {list(physio.columns)}")
physio.head()

In [ ]:
# ── Load surgical dataset ──────────────────────────────────────────────────────
if SURG_PATH.endswith('.csv'):
    surg = pd.read_csv(SURG_PATH, sep=None, engine='python', encoding='utf-8-sig')
else:
    surg = pd.read_excel(SURG_PATH)

# Standardise ID
def std_id(series):
    return series.astype(str).str.strip().str.split('.').str[0].str.upper()

surg[SURG_ID_COL] = std_id(surg[SURG_ID_COL])
surg = surg.rename(columns={SURG_ID_COL: PATIENT_ID})

print(f"Surgical dataset: {surg.shape}")
print(f"Outcome (compl) distribution:")
print(surg[OUTCOME_COL].value_counts(dropna=False))

In [ ]:
# ── Load ICF perioperative table (output of icf_perioperative_analysis.ipynb) ─
icf_peri = pd.read_csv(ICF_PERI_PATH)
icf_peri[PATIENT_ID] = std_id(icf_peri[PATIENT_ID])

ICF_PRE_COLS  = [c for c in icf_peri.columns if c.endswith('_pre')]
ICF_POST_COLS = [c for c in icf_peri.columns if c.endswith('_post')]

# Flag who has pre-op ICF
icf_peri['has_preop_icf'] = icf_peri[ICF_PRE_COLS].notna().any(axis=1)

print(f"ICF perioperative table: {icf_peri.shape}")
print(f"Pre-op ICF columns ({len(ICF_PRE_COLS)}): {ICF_PRE_COLS[:5]}...")
print(f"Patients with ≥1 pre-op ICF score: {icf_peri['has_preop_icf'].sum()}")

In [ ]:
# ── Standardise physio MDN ─────────────────────────────────────────────────────
if PATIENT_ID in physio.columns:
    physio[PATIENT_ID] = std_id(physio[PATIENT_ID])
else:
    physio = physio.rename(columns={physio.columns[0]: PATIENT_ID})
    physio[PATIENT_ID] = std_id(physio[PATIENT_ID])

if PHYSIO_RISK not in physio.columns:
    PHYSIO_RISK = physio.columns[1]
    print(f"Risk column auto-detected as: '{PHYSIO_RISK}'")

physio[PHYSIO_RISK] = physio[PHYSIO_RISK].astype(str).str.strip().str.lower()
print(f"Unique physio risk values: {physio[PHYSIO_RISK].value_counts().to_dict()}")

# Keep only esophagus patients from physio file
if DIAGNOSE_COL in physio.columns:
    physio_esoph = physio[physio[DIAGNOSE_COL].astype(str).str.lower().str.contains('esoph|oesoph', na=False)].copy()
    print(f"Physio esophagus patients: {len(physio_esoph)} (of {len(physio)} total)")
else:
    physio_esoph = physio.copy()
    print("No diagnose column found — using all physio patients")

# ── Build master table — keep ALL surg columns so Stage 4 preprocessing works ─
df = surg.copy()   # all columns from surgical dataset

# Merge physio conclusions
df = df.merge(physio_esoph[[PATIENT_ID, PHYSIO_RISK]].rename(columns={PHYSIO_RISK: 'physio_risk'}),
              on=PATIENT_ID, how='left')

# Merge ICF pre-op scores
df = df.merge(icf_peri[[PATIENT_ID, 'has_preop_icf'] + ICF_PRE_COLS],
              on=PATIENT_ID, how='left')

df['has_physio'] = df['physio_risk'].notna()
if 'has_preop_icf' not in df.columns:
    df['has_preop_icf'] = df[ICF_PRE_COLS].notna().any(axis=1)

print(f"\nMaster table: {df.shape[0]} patients x {df.shape[1]} columns")
print(f"Patients with physio conclusion:  {df['has_physio'].sum()}")
print(f"Patients with pre-op ICF scores:  {df['has_preop_icf'].sum()}")
print(f"Patients with BOTH:               {(df['has_physio'] & df['has_preop_icf']).sum()}")

In [ ]:
# ── Stage 4 preprocessing — mirrors staged_prediction_model.ipynb ─────────────
# Must run after merge-all so df has all surgical columns + ICF pre-op columns

# 1. Clean ASA sentinels
df['asascore_clean'] = pd.to_numeric(df['asascore'], errors='coerce')
df.loc[df['asascore_clean'] == 9, 'asascore_clean'] = np.nan

# 2. BMI
df['BMI'] = df['gewicht'] / ((df['lengte'] / 100) ** 2)
df.loc[df['BMI'] < 10,  'BMI'] = np.nan
df.loc[df['BMI'] > 60,  'BMI'] = np.nan

# 3. Age at operation
df['datok_dt']  = pd.to_datetime(df['datok'],  errors='coerce')
df['gebdat_dt'] = pd.to_datetime(df['gebdat'], errors='coerce')
df['age_at_op'] = (df['datok_dt'] - df['gebdat_dt']).dt.days / 365.25
df.loc[df['age_at_op'] < 18,  'age_at_op'] = np.nan
df.loc[df['age_at_op'] > 100, 'age_at_op'] = np.nan

# 4. CCI (same weights as staged model)
CCI_WEIGHTS = {
    'commyo': 1, 'comhartfaal': 1, 'comperif': 1, 'comcva': 1,
    'comlong': 1, 'comgiulcus': 1, 'comdiam': 1, 'comnier': 1, 'commalig': 6
}
cci_cols = [c for c in CCI_WEIGHTS if c in df.columns]
for c in cci_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
df['CCI'] = sum(df[c] * CCI_WEIGHTS[c] for c in cci_cols)

# 5. Cardiac comorbidities kept separate (not in CCI but clinically critical)
for c in ['comaf', 'comcarritme']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# 6. Lifestyle
for c in ['gewverl', 'dietist', 'anamok']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Smoking: Dutch text → ordinal (never=0, stopped=1, active=2)
if 'roken' in df.columns:
    def encode_smoking(val):
        v = str(val).strip().lower()
        if 'nooit' in v or 'never' in v: return 0
        if 'gestopt' in v or 'stop' in v: return 1
        if 'actief' in v or 'ja' in v or 'roker' in v: return 2
        return np.nan
    df['roken_enc']     = df['roken'].map(encode_smoking)
    df['roken_missing'] = df['roken_enc'].isna().astype(int)

# 7. Tumor staging
for c in ['scorect', 'scorecn']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# 8. Neoadjuvant
for c in ['neoadjaf', 'neoadjsch']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# 9. Surgical
for c in ['conversie', 'duur_operatie', 'bloedverlies']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
df['duur_operatie_missing'] = df['duur_operatie'].isna().astype(int) if 'duur_operatie' in df.columns else 0
df['bloedverlies_missing']  = df['bloedverlies'].isna().astype(int)  if 'bloedverlies'  in df.columns else 0

# 10. One-hot encode categoricals
CAT_COLS = ['tumlok1', 'tumhist', 'tumkeus', 'neoadjtype',
            'typok', 'aardok', 'procok', 'recontype', 'analig']
ohe_col_names = {}
ohe_dfs = []
for col in CAT_COLS:
    if col in df.columns:
        dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False).astype(int)
        ohe_col_names[col] = list(dummies.columns)
        ohe_dfs.append(dummies)
if ohe_dfs:
    df = pd.concat([df.reset_index(drop=True)] + [d.reset_index(drop=True) for d in ohe_dfs], axis=1)

# 11. Helper flags used in bias check
df['asa_ge3'] = (df['asascore_clean'] >= 3).astype(float)
df['year']    = df['datok_dt'].dt.year

# ── Build Stage 4 feature list ─────────────────────────────────────────────────
def get_existing(cols):
    return [c for c in cols if c in df.columns]

def get_ohe(col):
    return ohe_col_names.get(col, [])

STAGE1 = get_existing(['age_at_op', 'geslacht', 'asascore_clean', 'BMI'])
STAGE2 = STAGE1 + get_existing(['CCI', 'comaf', 'comcarritme',
                                  'roken_enc', 'roken_missing', 'gewverl', 'dietist', 'anamok'])
STAGE3 = STAGE2 + get_existing(['scorect', 'scorecn']) + \
         get_ohe('tumlok1') + get_ohe('tumhist') + get_ohe('tumkeus') + \
         get_existing(['neoadjaf']) + get_ohe('neoadjtype')
STAGE4 = list(dict.fromkeys(
    STAGE3 +
    get_existing(['conversie', 'duur_operatie', 'duur_operatie_missing',
                  'bloedverlies', 'bloedverlies_missing']) +
    get_ohe('typok') + get_ohe('aardok') + get_ohe('procok') +
    get_ohe('recontype') + get_ohe('analig')
))

print(f"Stage 1: {len(STAGE1)} features")
print(f"Stage 2: {len(STAGE2)} features")
print(f"Stage 3: {len(STAGE3)} features")
print(f"Stage 4: {len(STAGE4)} features  ← used as baseline in ICF analysis")
print(f"\nStage 4 feature list:\n{STAGE4}")

---
## Research Question 1 — Does the physiotherapist predict complications?

We encode the physio conclusion as ordinal: `none=0`, `low=1`, `high=2`  
Then treat it as a numeric score and compute AUC, sensitivity, specificity.

In [ ]:
# ── Encode physio risk ─────────────────────────────────────────────────────────
risk_map = {'none': 0, 'low': 1, 'high': 2}

# Try fuzzy matching in case values have extra whitespace or case issues
def encode_risk(val):
    val = str(val).strip().lower()
    for k, v in risk_map.items():
        if k in val:
            return v
    return np.nan

df['physio_score'] = df['physio_risk'].map(encode_risk)

print("Risk encoding:")
print(df[['physio_risk', 'physio_score']].dropna().groupby(['physio_risk', 'physio_score']).size())

In [ ]:
# ── Subset: patients with physio conclusion AND known outcome ──────────────────
physio_sub = df[df['physio_score'].notna() & df[OUTCOME_COL].notna()].copy()
N_physio = len(physio_sub)

print(f"Evaluation subset: N={N_physio}")
print(f"Complication rate in subset: {physio_sub[OUTCOME_COL].mean()*100:.1f}%")
print()

y_true  = physio_sub[OUTCOME_COL].astype(int)
y_score = physio_sub['physio_score']

auc = roc_auc_score(y_true, y_score)
ap  = average_precision_score(y_true, y_score)

print(f"Physiotherapist AUC-ROC:       {auc:.3f}")
print(f"Physiotherapist Avg Precision: {ap:.3f}")
print(f"Null model Brier:              {y_true.mean()*(1-y_true.mean()):.3f}")

In [ ]:
# ── Sensitivity / specificity at each threshold ────────────────────────────────
print("Performance at each risk category threshold:")
print(f"{'Threshold':<20} {'Sensitivity':>13} {'Specificity':>13} {'PPV':>8} {'NPV':>8} {'N flagged':>10}")
print("-" * 75)

for thresh_label, thresh_val in [("≥ low (any risk)", 1), ("≥ high only", 2)]:
    pred = (y_score >= thresh_val).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    n_flagged = pred.sum()
    print(f"{thresh_label:<20} {sens:>12.1%} {spec:>12.1%} {ppv:>7.1%} {npv:>7.1%} {n_flagged:>10}")

In [ ]:
# ── ROC curve of physiotherapist ──────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='#2266AA', linewidth=2.5,
             label=f'Physiotherapist  AUC={auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC — Physiotherapist risk classification', fontweight='bold')
axes[0].legend()

# Complication rate by risk group
comp_by_risk = physio_sub.groupby('physio_risk')[OUTCOME_COL].agg(['mean', 'count'])
comp_by_risk.columns = ['Complication rate', 'N']
comp_by_risk = comp_by_risk.loc[[k for k in ['none', 'low', 'high'] if k in comp_by_risk.index]]

bars = axes[1].bar(comp_by_risk.index, comp_by_risk['Complication rate'],
                   color=['#4CAF50', '#FFA726', '#E07050'], edgecolor='white', alpha=0.85)
for bar, (_, row) in zip(bars, comp_by_risk.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f"{row['Complication rate']:.1%}\n(n={int(row['N'])})",
                 ha='center', fontsize=10)
axes[1].set_ylabel('Complication rate')
axes[1].set_xlabel('Physiotherapist risk classification')
axes[1].set_title('Observed complication rate by physio risk group', fontweight='bold')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## Research Question 2 — Selection bias: are ICF patients different?

We compare ICF-available (N≈533) vs ICF-missing patients on key preoperative characteristics.  
If they differ significantly, any improvement from adding ICF scores must be interpreted cautiously.

In [ ]:
# All computed in preprocessing cell — just confirm columns exist
print("Columns available for bias check:")
for col in ['age_at_op', 'BMI', 'asascore_clean', 'asa_ge3', 'neoadjaf', OUTCOME_COL, 'year']:
    present = col in df.columns
    print(f"  {col:<25} {'✓' if present else '✗ MISSING'}")

In [ ]:
# ── Compare ICF-available vs ICF-missing ─────────────────────────────────────
grp_icf   = df[df['has_preop_icf'] == True]
grp_noicf = df[df['has_preop_icf'] == False]

print(f"ICF-available:  N={len(grp_icf)}")
print(f"ICF-missing:    N={len(grp_noicf)}")
print()

rows = []

continuous_vars = [
    ('age_at_op',      'Age (years)'),
    ('BMI',            'BMI (kg/m²)'),
    ('asascore_clean', 'ASA score'),
    ('CCI',            'CCI score'),
]
for col, label in continuous_vars:
    if col not in df.columns:
        continue
    a = grp_icf[col].dropna()
    b = grp_noicf[col].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    rows.append({
        'Variable':      label,
        'ICF-available': f"{a.median():.1f} (IQR {a.quantile(0.25):.1f}–{a.quantile(0.75):.1f})",
        'ICF-missing':   f"{b.median():.1f} (IQR {b.quantile(0.25):.1f}–{b.quantile(0.75):.1f})",
        'Test':    'Mann-Whitney',
        'p-value': round(p, 4),
        'Sig':     '✓' if p < 0.05 else '–'
    })

binary_vars = [
    (OUTCOME_COL, 'Complication (%)'),
    ('asa_ge3',   'ASA ≥ III (%)'),
    ('neoadjaf',  'Neoadjuvant (%)'),
]
for col, label in binary_vars:
    if col not in df.columns:
        continue
    a = grp_icf[col].dropna()
    b = grp_noicf[col].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    ct = pd.crosstab(df['has_preop_icf'], df[col].round(0))
    if ct.shape == (2, 2):
        _, p, _, _ = stats.chi2_contingency(ct)
    else:
        p = np.nan
    rows.append({
        'Variable':      label,
        'ICF-available': f"{a.mean()*100:.1f}%",
        'ICF-missing':   f"{b.mean()*100:.1f}%",
        'Test':    'Chi-square',
        'p-value': round(p, 4) if not np.isnan(p) else 'n/a',
        'Sig':     '✓' if (not np.isnan(p) and p < 0.05) else '–'
    })

bias_table = pd.DataFrame(rows)
print("Selection Bias Check — ICF-available vs ICF-missing:")
print("=" * 90)
print(bias_table.to_string(index=False))
print("=" * 90)
print("\n✓ = statistically significant difference (p < 0.05)")

In [ ]:
# ── Year distribution — are ICF notes concentrated in recent years? ────────────
if 'year' in df.columns:
    year_tbl = df.groupby('year')['has_preop_icf'].agg(['sum', 'count'])
    year_tbl.columns = ['N with ICF', 'N total']
    year_tbl['ICF coverage %'] = (year_tbl['N with ICF'] / year_tbl['N total'] * 100).round(1)
    print("ICF coverage by year of surgery:")
    print(year_tbl)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(year_tbl.index, year_tbl['N total'], color='#cccccc', label='Total', alpha=0.8)
    ax.bar(year_tbl.index, year_tbl['N with ICF'], color='#2266AA', label='Has pre-op ICF', alpha=0.85)
    ax.set_xlabel('Year of surgery')
    ax.set_ylabel('Number of patients')
    ax.set_title('ICF data availability by year', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Research Question 3 — Do preoperative ICF scores add predictive value?

**Strategy:** Restrict to the ICF-available subset (N≈533).  
Within that subset, compare:
- **Baseline:** Logistic Regression + RF on Stage 4 registry features only  
- **+ICF:** Same models with preoperative ICF scores added as extra features

This isolates whether ICF adds value *beyond* what the registry already captures.

In [ ]:
# ── Build feature sets — Stage 4 baseline + ICF ───────────────────────────────
# STAGE4 already built in preprocessing cell (39 features, mirrors main model)

icf_sub = df[df['has_preop_icf'] == True].copy()

icf_coverage = icf_sub[ICF_PRE_COLS].notna().mean()

# Only keep ICF columns where >50% of the ICF subset has actual recorded values.
# Columns below this threshold are too sparse — we do NOT impute ICF scores.
ICF_USEABLE = icf_coverage[icf_coverage > 0.50].index.tolist()

print(f"ICF subset size: N={len(icf_sub)}")
print(f"Outcome rate in ICF subset: {icf_sub[OUTCOME_COL].mean()*100:.1f}%")
print(f"\nStage 4 baseline: {len(STAGE4)} features")
print(f"\nAll ICF pre-op columns — coverage in ICF subset:")
for c in sorted(ICF_PRE_COLS, key=lambda x: -icf_coverage[x]):
    n   = icf_sub[c].notna().sum()
    pct = n / len(icf_sub) * 100
    flag = '✓ KEPT' if c in ICF_USEABLE else '✗ dropped (<50%)'
    print(f"  {c:<35} {n:>4} / {len(icf_sub)}  ({pct:.0f}%)  {flag}")

print(f"\nICF columns used in model: {len(ICF_USEABLE)}")
print(ICF_USEABLE)

In [ ]:
from sklearn.impute import SimpleImputer

# Drop rows missing outcome
icf_sub = icf_sub[icf_sub[OUTCOME_COL].notna()].copy()

# Stage 4 features actually present in df
S4 = [f for f in STAGE4 if f in icf_sub.columns]

# Complete-case subset: patients who have actual values for all kept ICF columns
icf_complete = icf_sub.dropna(subset=ICF_USEABLE).copy().reset_index(drop=True)

print(f"ICF subset (≥1 pre-op ICF note):       N={len(icf_sub)}")
print(f"Complete-case (all kept ICF cols):      N={len(icf_complete)}  "
      f"({len(icf_complete)/len(icf_sub)*100:.0f}% of ICF subset)")
print(f"Outcome rate in complete-case subset:   {icf_complete[OUTCOME_COL].mean()*100:.1f}%")
print(f"Stage 4 features used as baseline:      {len(S4)}")

# NOTE: imputation (median, Stage 4 features only) and scaling are NOT fit here.
# To prevent leakage, both are fit fresh inside each CV fold via sklearn Pipeline
# (see icf-cv cell below) rather than once on the full complete-case set.

# Raw (unimputed, unscaled) feature matrices — imputer/scaler live inside the CV pipeline
X_reg_cc   = icf_complete[S4].reset_index(drop=True)

# ICF features: NO imputation — only real recorded values
X_icf_cc   = icf_complete[ICF_USEABLE].reset_index(drop=True)

X_icf_full = pd.concat([X_reg_cc, X_icf_cc], axis=1)
y_cc       = icf_complete[OUTCOME_COL].astype(int).values

print(f"\nX_baseline (Stage 4 only):        {X_reg_cc.shape}")
print(f"X_baseline + ICF (real values):   {X_icf_full.shape}")
print(f"y: {len(y_cc)} patients, {y_cc.mean()*100:.1f}% complications")

In [ ]:
# ── Cross-validated AUC: Stage 4 baseline vs Stage 4 + ICF ───────────────────
# Leakage prevention: imputer (median, registry features only) and scaler are fit
# fresh on each CV training fold via Pipeline — never on the full N=132 set up front.
from sklearn.pipeline import Pipeline

scoring = {
    'roc_auc':       'roc_auc',
    'avg_precision': 'average_precision',
    'brier':         make_scorer(brier_score_loss, response_method='predict_proba',
                                  greater_is_better=False)
}

lr_params = dict(max_iter=2000, random_state=RANDOM_SEED)
rf_params = dict(n_estimators=300, max_depth=5, min_samples_leaf=10,
                 random_state=RANDOM_SEED, n_jobs=-1)

def make_lr_pipeline():
    return Pipeline([
        ("impute", SimpleImputer(strategy='median')),
        ("scale",  StandardScaler()),
        ("model",  LogisticRegression(**lr_params)),
    ])

def make_rf_pipeline():
    # RF doesn't need scaling, but still needs imputation fit per-fold
    return Pipeline([
        ("impute", SimpleImputer(strategy='median')),
        ("model",  RandomForestClassifier(**rf_params)),
    ])

comparisons = {
    'LR — Stage 4 only':      (make_lr_pipeline(), X_reg_cc,   y_cc),
    'LR — Stage 4 + ICF':     (make_lr_pipeline(), X_icf_full, y_cc),
    'RF — Stage 4 only':      (make_rf_pipeline(), X_reg_cc,   y_cc),
    'RF — Stage 4 + ICF':     (make_rf_pipeline(), X_icf_full, y_cc),
}

cv_rows = []
for name, (pipeline, X, y) in comparisons.items():
    res = cross_validate(pipeline, X, y, cv=CV, scoring=scoring, return_train_score=False)
    cv_rows.append({
        'Model':       name,
        'CV AUC':      f"{res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std():.3f}",
        'CV Avg Prec': f"{res['test_avg_precision'].mean():.3f} ± {res['test_avg_precision'].std():.3f}",
        'CV Brier':    f"{-res['test_brier'].mean():.3f} ± {res['test_brier'].std():.3f}",
        'AUC_raw':     res['test_roc_auc'].mean(),
        'AUC_folds':   res['test_roc_auc']
    })
    print(f"{name:<30}  AUC = {res['test_roc_auc'].mean():.3f} ± {res['test_roc_auc'].std():.3f}")

print(f"\nComplete-case N={len(y_cc)}, prevalence={y_cc.mean()*100:.1f}%")
print(f"Null Brier: {y_cc.mean()*(1-y_cc.mean()):.3f}")

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
cv_table = pd.DataFrame([{k: v for k, v in row.items() if k not in ('AUC_raw', 'AUC_folds')}
                         for row in cv_rows])

print(f"5-Fold CV Results — complete-case ICF subset (N={len(y_cc)}, {y_cc.mean()*100:.1f}% complications):")
print("=" * 80)
print(cv_table.to_string(index=False))
print("=" * 80)

lr_base = next(r for r in cv_rows if r['Model'] == 'LR — Stage 4 only')['AUC_raw']
lr_icf  = next(r for r in cv_rows if r['Model'] == 'LR — Stage 4 + ICF')['AUC_raw']
rf_base = next(r for r in cv_rows if r['Model'] == 'RF — Stage 4 only')['AUC_raw']
rf_icf  = next(r for r in cv_rows if r['Model'] == 'RF — Stage 4 + ICF')['AUC_raw']

print(f"\nΔ AUC from adding ICF scores (same N, complete-case):")
print(f"  Logistic Regression: {lr_icf - lr_base:+.3f}")
print(f"  Random Forest:       {rf_icf - rf_base:+.3f}")

In [ ]:
# ── AUC boxplot: registry vs registry+ICF ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

auc_data    = [r['AUC_folds'] for r in cv_rows]
model_names = [r['Model'] for r in cv_rows]
colors_bp   = ['#88BBDD', '#2266AA', '#FFAA66', '#E07050']

bp = ax.boxplot(auc_data, labels=model_names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.axhline(0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='AUC = 0.70')
ax.set_ylabel('AUC-ROC')
ax.set_title(f'5-Fold CV AUC: Registry vs Registry + ICF  (N={len(y_cc)}, complete-case)', fontweight='bold')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── LASSO on registry + ICF to see which domains survive regularisation ────────
# This is a single descriptive fit on the full complete-case set (not a CV
# generalisation estimate), so impute+scale once here is appropriate — the
# leakage fix above applies to the CV performance comparison, not this step.
imp_final   = SimpleImputer(strategy='median')
X_reg_imp   = pd.DataFrame(imp_final.fit_transform(X_icf_full[S4]), columns=S4)
X_icf_final = pd.concat([X_reg_imp, X_icf_full[ICF_USEABLE].reset_index(drop=True)], axis=1)

scaler_final  = StandardScaler()
X_icf_scaled  = scaler_final.fit_transform(X_icf_final)

lasso = LogisticRegressionCV(
    Cs=np.logspace(-3, 2, 20),
    cv=CV, penalty='l1', solver='liblinear',
    scoring='roc_auc', max_iter=2000, random_state=RANDOM_SEED
)
lasso.fit(X_icf_scaled, y_cc)

all_features = S4 + ICF_USEABLE
coef_df = pd.DataFrame({
    'Feature':     all_features,
    'Coefficient': lasso.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

kept    = coef_df[coef_df['Coefficient'] != 0]
dropped = coef_df[coef_df['Coefficient'] == 0]

print(f"LASSO optimal C: {lasso.C_[0]:.4f}")
print(f"Non-zero features: {len(kept)} / {len(all_features)}")
print(f"Dropped to zero:   {dropped['Feature'].tolist()}")
print()
print(kept.to_string(index=False))

surviving_icf = [f for f in ICF_USEABLE if f in kept['Feature'].values]
print(f"\nICF domains that survive LASSO: {surviving_icf if surviving_icf else 'none'}")

In [ ]:
# ── LASSO coefficient plot — English labels, trimmed to |coef| > 0.05 ─────────
if len(kept) > 0:

    LABEL_MAP = {
        'procok_3.0':            'Approach: robotic',
        'procok_2.0':            'Approach: hybrid',
        'procok_1.0':            'Approach: open thoracic',
        'procok_4.0':            'Approach: MIE',
        'procok_9.0':            'Approach: other',
        'duur_operatie':         'Operation duration',
        'duur_operatie_missing': 'Op. duration: not recorded',
        'bloedverlies':          'Intraop. blood loss',
        'bloedverlies_missing':  'Blood loss: not recorded',
        'analig_3.0':            'Anastomosis: abdominal',
        'analig_2.0':            'Anastomosis: intrathoracic',
        'analig_1.0':            'Anastomosis: cervical',
        'analig_0.0':            'Anastomosis: none',
        'analig_9.0':            'Anastomosis: other',
        'conversie':             'Conversion to open',
        'neoadjtype_1.0':        'Neoadjuvant: chemotherapy',
        'neoadjtype_2.0':        'Neoadjuvant: chemoradiotherapy',
        'neoadjtype_3.0':        'Neoadjuvant: radiotherapy',
        'neoadjtype_4.0':        'Neoadjuvant: other',
        'age_at_op':             'Age at surgery',
        'geslacht':              'Sex (male)',
        'BMI':                   'BMI',
        'asascore_clean':        'ASA score',
        'gewverl':               'Preop. weight loss',
        'CCI':                   'Charlson Comorbidity Index',
        'roken_enc':             'Smoking status',
        'roken_missing':         'Smoking: unknown',
        'comaf':                 'Atrial fibrillation',
        'comcarritme':           'Cardiac arrhythmia',
        'INS-B455_lvl_pre':      'Exercise tolerance (INS-B455)',
        'FML-D760_lvl_pre':      'Family functioning (FML-D760)',
        'MBW-B530_lvl_pre':      'Body weight maint. (MBW-B530)',
        'FAC-D540_lvl_pre':      'Self-care (FAC-D540)',
        'ETN-D550_lvl_pre':      'Eating (ETN-D550)',
        'ADM-B440_lvl_pre':      'Respiration (ADM-B440)',
    }

    # Keep only features with meaningful coefficient size
    plot_df = kept[kept['Coefficient'].abs() > 0.05].sort_values('Coefficient').copy()
    plot_df['Label'] = plot_df['Feature'].map(LABEL_MAP).fillna(plot_df['Feature'])
    is_icf  = plot_df['Feature'].isin(ICF_USEABLE)
    colors  = ['#9C27B0' if icf else ('#E07050' if c < 0 else '#4CAF50')
               for icf, c in zip(is_icf, plot_df['Coefficient'])]

    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.45)))
    ax.barh(plot_df['Label'], plot_df['Coefficient'],
            color=colors, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Standardised LASSO Coefficient')
    ax.set_title(f'LASSO — Stage 4 Registry + ICF  (|coef| > 0.05 shown, n={len(plot_df)} of {len(kept)} features)',
                 fontweight='bold', fontsize=11)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#9C27B0', label='ICF domain (preoperative)'),
        Patch(facecolor='#4CAF50', label='Registry feature (↑ risk)'),
        Patch(facecolor='#E07050', label='Registry feature (↓ risk)'),
    ]
    ax.legend(handles=legend_elements, fontsize=9)
    plt.tight_layout()

    import os
    save_dir = os.getcwd()  # saves next to the notebook wherever it runs
    plt.savefig(os.path.join(save_dir, 'predicting_features_icf.png'), dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {os.path.join(save_dir, 'predicting_features_icf.png')}  ({len(plot_df)} features shown)")

---
## Summary

| Question | Finding |
|---|---|
| Does the physiotherapist predict complications? | AUC = *see above* |
| Are ICF-available patients a biased subset? | *See selection bias table* |
| Do preoperative ICF scores add value to the model? | ΔAUC LR = *see above*, ΔAUC RF = *see above* |

**Interpretation notes:**
- If selection bias is present (ICF-available patients differ on key variables), interpret any ΔAUC cautiously — the improvement may reflect the patient mix, not the ICF information itself.
- LASSO identifies which specific ICF domains carry signal; domains shrunk to zero do not add independent predictive value.
- The physiotherapist AUC can be benchmarked against the staged ML model AUC on the same subset for a direct comparison.